# `parse_pptx` — extraction d'un .pptx en DataFrames

Module testé : [`src/docpipeline/parsing/pptx/parse_pptx.py`](../../../src/docpipeline/parsing/pptx/parse_pptx.py).

## Sortie de `parse_pptx(pptx)`

| Sortie | Contenu |
|---|---|
| `slide_df`     | 1 ligne = 1 slide (layout, n_shapes, n_runs, n_images, n_tables, has_notes) |
| `shape_df`     | 1 ligne = 1 shape (slide, type, position, dimensions, has_text/table/image) |
| `paragraph_df` | 1 ligne = 1 paragraphe de texte (slide, shape, paragraph_index, text, level, alignment) |
| `runs_df`      | 1 ligne = 1 run (`span_id` `pp_<slide>_<shape>_<para>_<run>`, text, font_name, font_size, bold, italic, underline, color, hyperlink) |
| `image_df`     | 1 ligne = 1 image embarquée (position, dimensions) |
| `table_df`     | 1 ligne = 1 table (n_rows, n_cols) |
| `doc_summary`  | JSON synthèse : metadata, source_tool, n_slides, n_runs_bold/italic/with_color, has_speaker_notes, layout_counts |

Le `span_id` (format `pp_<slide>_<shape>_<para>_<run>`) est stable et reproductible — clé pour un futur cycle extract → modify → rebuild (translation PPTX).

## Démo

Le notebook crée d'abord une fixture `tests/fixtures/contrat_assurance.pptx` (4 slides : titre, bullets, paragraphe avec runs mixtes, table + speaker notes), puis l'extrait avec `parse_pptx`. Pas de LLM (règle CLAUDE.md).

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ../../..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
# 1) Génération de la fixture .pptx (si absente)
from pathlib import Path
from pptx import Presentation
from pptx.util import Inches
from pptx.dml.color import RGBColor

FIXTURE = Path('../../../tests/fixtures/contrat_assurance.pptx')
FIXTURE.parent.mkdir(parents=True, exist_ok=True)

if not FIXTURE.exists():
    prs = Presentation()
    prs.slide_width  = Inches(10)
    prs.slide_height = Inches(7.5)

    # Slide 1 : Title
    s1 = prs.slides.add_slide(prs.slide_layouts[0])
    s1.shapes.title.text = "Contrat d'assurance"
    s1.placeholders[1].text = "Conditions G\u00e9n\u00e9rales \u2014 Pr\u00e9sentation 2026"

    # Slide 2 : Bullets
    s2 = prs.slides.add_slide(prs.slide_layouts[1])
    s2.shapes.title.text = "1. Garanties principales"
    body = s2.placeholders[1].text_frame
    body.text = "Garantie IA (Individual Accident)"
    body.add_paragraph().text = "Garantie BI (Business Interruption)"
    body.add_paragraph().text = "Responsabilit\u00e9 Civile (RC)"
    p = body.add_paragraph(); p.text = "Plafond global : 200 000 EUR"
    for r in p.runs:
        r.font.bold = True
        r.font.color.rgb = RGBColor(0xC0, 0x39, 0x2B)

    # Slide 3 : runs mixtes (bold + italic + color)
    s3 = prs.slides.add_slide(prs.slide_layouts[5])
    s3.shapes.title.text = "2. Franchise et d\u00e9lais"
    tb = s3.shapes.add_textbox(Inches(0.5), Inches(1.5), Inches(9), Inches(2))
    tf = tb.text_frame; tf.word_wrap = True
    p = tf.paragraphs[0]
    p.add_run().text = "La franchise applicable est de "
    r = p.add_run(); r.text = "300 EUR par sinistre"; r.font.bold = True
    p.add_run().text = ". Toute d\u00e9claration doit intervenir dans un d\u00e9lai de "
    r = p.add_run(); r.text = "5 jours ouvr\u00e9s"; r.font.italic = True; r.font.color.rgb = RGBColor(0x1F, 0x49, 0x7D)
    p.add_run().text = "."
    p2 = tf.add_paragraph()
    r = p2.add_run(); r.text = "IMPORTANT : "; r.font.bold = True; r.font.color.rgb = RGBColor(0xC0, 0x39, 0x2B)
    p2.add_run().text = "le non-respect de ce d\u00e9lai entra\u00eene la d\u00e9ch\u00e9ance de garantie."

    # Slide 4 : Tableau + speaker notes
    s4 = prs.slides.add_slide(prs.slide_layouts[5])
    s4.shapes.title.text = "3. Tableau des garanties"
    table = s4.shapes.add_table(4, 3, Inches(0.5), Inches(1.5), Inches(9), Inches(2.5)).table
    table.cell(0,0).text = "Garantie";                 table.cell(0,1).text = "Plafond";        table.cell(0,2).text = "Franchise"
    table.cell(1,0).text = "IA \u2014 Individual Accident";    table.cell(1,1).text = "50 000 EUR";     table.cell(1,2).text = "300 EUR"
    table.cell(2,0).text = "BI \u2014 Business Interruption";  table.cell(2,1).text = "100 000 EUR";    table.cell(2,2).text = "1 000 EUR"
    table.cell(3,0).text = "RC \u2014 Responsabilit\u00e9 Civile";  table.cell(3,1).text = "200 000 EUR";    table.cell(3,2).text = "0 EUR"
    s4.notes_slide.notes_text_frame.text = "Cf. article 5 du contrat pour le d\u00e9tail des plafonds."

    prs.save(str(FIXTURE))
    print(f'OK fixture creee : {FIXTURE} ({FIXTURE.stat().st_size} bytes)')
else:
    print(f'fixture existe deja : {FIXTURE} ({FIXTURE.stat().st_size} bytes)')

fixture existe deja : ..\tests\fixtures\contrat_assurance.pptx (36289 bytes)


In [3]:
# 2) Extraction
import pandas as pd
from IPython.display import display, Markdown
from docpipeline.parsing.pptx.parse_pptx import parse_pptx

pd.set_option('display.max_colwidth', 100)

result = parse_pptx(FIXTURE)

display(Markdown('### 1. `doc_summary` — synthèse au niveau document'))
summary_df = pd.DataFrame([{'key': k, 'value': str(v)} for k, v in result['doc_summary'].items()])
display(summary_df)

display(Markdown(f"### 2. `slide_df` — {len(result['slide_df'])} slides"))
display(result['slide_df'])

display(Markdown(f"### 3. `runs_df` — {len(result['runs_df'])} runs avec leurs styles"))
display(result['runs_df'][['span_id', 'slide_index', 'text', 'font_name', 'font_size_pt', 'bold', 'italic', 'underline', 'color']])

display(Markdown(f"### 4. `paragraph_df` — {len(result['paragraph_df'])} paragraphes"))
display(result['paragraph_df'][['slide_index', 'shape_index', 'paragraph_index', 'text', 'level', 'alignment', 'n_runs']])

display(Markdown(f"### 5. `shape_df` — {len(result['shape_df'])} shapes"))
display(result['shape_df'][['slide_index', 'shape_index', 'shape_type', 'has_text', 'has_table', 'is_picture', 'width_pt', 'height_pt']])

display(Markdown(f"### 6. `table_df` — {len(result['table_df'])} table(s) native(s)"))
display(result['table_df'])

### 1. `doc_summary` — synthèse au niveau document

,key,value
0,doc_hash,6217b8d7ef5fd84835a060a6aa986bfbe0a8bebe3f9e248a775267f4d734c4fd
1,file_size_bytes,36289
2,n_slides,4
3,n_shapes,8
4,n_text_runs,16
5,n_images,0
6,n_tables,1
7,total_char_count,443
8,source_tool,Microsoft PowerPoint
9,metadata,"{'title': '', 'author': '', 'last_modified_by': 'Steve Canny', 'subject': '', 'keywords': '', 'c..."


### 2. `slide_df` — 4 slides

,doc_hash,slide_index,layout_name,n_shapes,n_text_runs,n_images,n_tables,has_notes,notes_text
0,6217b8d7ef5fd84835a060a6aa986bfbe0a8bebe3f9e248a775267f4d734c4fd,0,Title Slide,2,2,0,0,False,
1,6217b8d7ef5fd84835a060a6aa986bfbe0a8bebe3f9e248a775267f4d734c4fd,1,Title and Content,2,5,0,0,False,
2,6217b8d7ef5fd84835a060a6aa986bfbe0a8bebe3f9e248a775267f4d734c4fd,2,Title Only,2,8,0,0,False,
3,6217b8d7ef5fd84835a060a6aa986bfbe0a8bebe3f9e248a775267f4d734c4fd,3,Title Only,2,1,0,1,True,Cf. article 5 du contrat pour le détail des plafonds.


### 3. `runs_df` — 16 runs avec leurs styles

...

,slide_index,shape_index,paragraph_index,text,level,alignment,n_runs
0,0,0,0,Contrat d'assurance,0,None,1
1,0,1,0,Conditions Générales — Présentation 2026,0,None,1
2,1,0,0,1. Garanties principales,0,None,1
3,1,1,0,Garantie IA (Individual Accident),0,None,1
4,1,1,1,Garantie BI (Business Interruption),0,None,1
5,1,1,2,Responsabilité Civile (RC),0,None,1
6,1,1,3,Plafond global : 200 000 EUR,0,None,1
7,2,0,0,2. Franchise et délais,0,None,1
8,2,1,0,La franchise applicable est de 300 EUR par sinistre. Toute déclaration doit intervenir dans un d...,0,None,5
9,2,1,1,IMPORTANT : le non-respect de ce délai entraîne la déchéance de garantie.,0,None,2


### 5. `shape_df` — 8 shapes

,slide_index,shape_index,shape_type,has_text,has_table,is_picture,width_pt,height_pt
0,0,0,PLACEHOLDER,True,False,False,612.0,115.75
1,0,1,PLACEHOLDER,True,False,False,504.0,138.00
2,1,0,PLACEHOLDER,True,False,False,648.0,90.00
3,1,1,PLACEHOLDER,True,False,False,648.0,356.38
4,2,0,PLACEHOLDER,True,False,False,648.0,90.00
5,2,1,TEXT_BOX,True,False,False,648.0,144.00
6,3,0,PLACEHOLDER,True,False,False,648.0,90.00
7,3,1,TABLE,False,True,False,648.0,180.00


### 6. `table_df` — 1 table(s) native(s)

,doc_hash,table_index,slide_index,shape_index,n_rows,n_cols,n_cells_with_text
0,6217b8d7ef5fd84835a060a6aa986bfbe0a8bebe3f9e248a775267f4d734c4fd,0,3,1,4,3,12
